# Global Commodity & Currency Market Analysis (1960–2026)

## Base Notebook – Data Loading & Initial Analysis (CSV Source)

**Note:** The cell below expects `Brent_cleaned_updated.csv`, `gold_cleaned_updated.csv`, and `silver_cleaned_updated.csv` at the original Kaggle input path. These raw CSV files were not included in the uploaded project files, so this section will raise a `FileNotFoundError` unless the three CSVs are placed at that path (or the path is updated to point to them). Code is preserved as-is per project instructions — no placeholder data was generated.

In [ ]:
import pandas as pd

# Load all three datasets
brent = pd.read_csv('/kaggle/input/datasets/prarthanap374/commodity-price-analysis/Brent_cleaned_updated.csv')
gold  = pd.read_csv('/kaggle/input/datasets/prarthanap374/commodity-price-analysis/gold_cleaned_updated.csv')
silver= pd.read_csv('/kaggle/input/datasets/prarthanap374/commodity-price-analysis/silver_cleaned_updated.csv')

# Preview each one
print("BRENT OIL:")
print(brent.head())
print("\nGOLD:")
print(gold.head())
print("\nSILVER:")
print(silver.head()) 

In [ ]:
highest_price = brent['Price'].max()
lowest_price = brent['Price'].min()

highest_year = brent.loc[brent['Price'].idxmax(), 'Year']
lowest_year = brent.loc[brent['Price'].idxmin(), 'Year']

overall_avg = brent['Price'].mean()

print("====== Q6: BRENT OIL ======")
print(f"Highest Price : ${highest_price:.2f} (Year: {highest_year})")
print(f"Lowest Price : ${lowest_price:.2f} (Year: {lowest_year})")
print(f"Overall Average: ${overall_avg:.2f}")                         

In [ ]:
print("====== Q7: GOLD ======")
print(f"Highest Price : ${gold['Price'].max():.2f} (Year: {gold.loc[gold['Price'].idxmax(), 'Year']})")
print(f"Lowest Price : ${gold['Price'].min():.2f} (Year: {gold.loc[gold['Price'].idxmin(), 'Year']})")
print(f"Overall Average: ${gold['Price'].mean():.2f}")

In [ ]:
print("====== Q8: SILVER ======")
print(f"Highest Price : ${silver['Price'].max():.2f} (Year: {silver.loc[silver['Price'].idxmax(), 'Year']})")
print(f"Lowest Price : ${silver['Price'].min():.2f} (Year: {silver.loc[silver['Price'].idxmin(), 'Year']})")
print(f"Overall Average: ${silver['Price'].mean():.2f}")

In [ ]:
brent_avg = brent.groupby('Year')['Price'].mean().rename('Brent_Avg')
gold_avg = gold.groupby('Year')['Price'].mean().rename('Gold_Avg')
silver_avg = silver.groupby('Year')['Price'].mean().rename('Silver_Avg')

annual = pd.concat([brent_avg, gold_avg, silver_avg], axis=1).round(2)
annual.index.name = 'Year'

print("====== Q9: ANNUAL AVERAGES ======")
print(annual.to_string())

In [ ]:
top5_brent = annual['Brent_Avg'].nlargest(5).reset_index()
top5_brent.columns = ['Year', 'Avg Brent Price ($/bbl)']
top5_brent.index = top5_brent.index + 1
top5_brent.index.name = 'Rank'

print("====== Q10: TOP 5 BRENT OIL YEARS ======")
print(top5_brent.to_string())

In [ ]:
top5_gold = annual['Gold_Avg'].nlargest(5).reset_index()
top5_gold.columns = ['Year', 'Avg Gold Price ($/oz)']
top5_gold.index = top5_gold.index + 1
top5_gold.index.name = 'Rank'

print("====== Q10: TOP 5 GOLD OIL YEARS ======")
print(top5_gold.to_string())

In [ ]:
top5_silver = annual['Silver_Avg'].nlargest(5).reset_index()
top5_silver.columns = ['Year', 'Avg Silver Price ($/oz)']
top5_silver.index = top5_silver.index + 1
top5_silver.index.name = 'Rank'

print("====== Q10: TOP 5 SILVER OIL YEARS ======")
print(top5_silver.to_string())

In [ ]:
annual['Brent_%Chg'] = annual['Brent_Avg'].pct_change() * 100
annual['Gold_%Chg'] = annual['Gold_Avg'].pct_change() * 100
annual['Silver_%Chg'] = annual['Silver_Avg'].pct_change() * 100

annual = annual.round(2)

print("====== BONUS B1: YEARLY % CHANGE ======")
print(annual[['Brent_%Chg', 'Gold_%Chg', 'Silver_%Chg']].to_string())

## Shared Utilities – data_utils.py

In [ ]:
"""
data_utils.py
-------------
Shared helper module for the Global Commodity & Currency Market Research report.

Builds monthly time series (1960-2026) for Brent Crude Oil, Gold, and Silver
by interpolating between well-known historical benchmark prices and adding
realistic month-to-month noise. Also provides the small static tables used
for the currency-share and offshore-RMB charts.

NOTE: These series are reconstructed approximations for report/chart-building
purposes (anchored on well-documented historical price levels), not an
official data feed. Replace `load_real_data()` with a call to your own
CSV/Excel/Refinitiv/World-Bank pull if you have the original source file.
"""

import numpy as np
import pandas as pd


def _build_series(anchor_dates, anchor_prices, freq="MS", noise_pct=0.03, seed=1):
    """Interpolate anchor points onto a monthly index and add smooth noise."""
    anchors = pd.Series(anchor_prices, index=pd.to_datetime(anchor_dates))
    full_index = pd.date_range(anchors.index.min(), anchors.index.max(), freq=freq)
    series = anchors.reindex(full_index)
    series = series.interpolate(method="pchip").clip(lower=0.01)

    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0, scale=noise_pct, size=len(series))
    # smooth the noise a little so months aren't independent random jumps
    noise = pd.Series(noise, index=series.index).rolling(3, min_periods=1).mean()
    series = series * (1 + noise)
    return series.round(2)


def get_brent_series():
    dates = ["1960-01-01", "1973-09-01", "1974-01-01", "1979-06-01", "1980-06-01",
              "1986-01-01", "1986-06-01", "1990-06-01", "1990-10-01", "1993-01-01",
              "1999-01-01", "2000-09-01", "2004-01-01", "2008-07-01", "2008-12-01",
              "2011-04-01", "2012-03-01", "2014-06-01", "2016-01-01", "2018-10-01",
              "2020-04-01", "2021-06-01", "2022-06-01", "2023-06-01", "2025-01-01",
              "2026-06-01"]
    prices = [1.6, 3, 11, 15, 40, 27, 10, 17, 34, 17, 10, 33, 30, 133, 42,
              123, 125, 108, 30, 80, 24, 73, 120, 76, 72, 121]
    return _build_series(dates, prices, noise_pct=0.05, seed=10)


def get_gold_series():
    dates = ["1960-01-01", "1971-08-01", "1974-01-01", "1979-01-01", "1980-01-01",
              "1982-01-01", "1987-01-01", "1996-01-01", "2001-01-01", "2005-01-01",
              "2008-10-01", "2011-09-01", "2012-10-01", "2015-12-01", "2018-08-01",
              "2020-08-01", "2022-09-01", "2024-01-01", "2024-10-01", "2026-06-01"]
    prices = [35, 43, 155, 500, 675, 375, 480, 390, 270, 445, 730, 1780, 1750, 1060,
               1195, 2000, 1650, 2050, 2700, 5030]
    return _build_series(dates, prices, noise_pct=0.03, seed=20)


def get_silver_series():
    dates = ["1960-01-01", "1974-01-01", "1979-06-01", "1980-01-01", "1980-06-01",
              "1982-06-01", "1987-01-01", "1993-01-01", "2001-01-01", "2006-01-01",
              "2008-10-01", "2011-04-01", "2013-06-01", "2015-12-01", "2020-03-01",
              "2021-02-01", "2022-09-01", "2024-01-01", "2025-06-01", "2026-06-01"]
    prices = [0.9, 5, 8, 39, 16, 6, 7, 3.7, 4.4, 10.5, 9, 42, 21, 14, 12,
               27, 19, 24, 34, 92]
    return _build_series(dates, prices, noise_pct=0.06, seed=30)


def get_usd_cny_series():
    dates = ["1994-01-01", "2005-07-01", "2008-01-01", "2014-01-01", "2016-01-01",
              "2018-06-01", "2020-05-01", "2022-10-01", "2024-01-01", "2026-06-01"]
    rates = [8.7, 8.1, 6.83, 6.05, 6.95, 6.4, 7.1, 7.3, 7.2, 7.1]
    return _build_series(dates, rates, noise_pct=0.01, seed=40)


def get_market_frame():
    """Combine Brent / Gold / Silver into one aligned monthly DataFrame (1960-2026)."""
    brent = get_brent_series()
    gold = get_gold_series()
    silver = get_silver_series()
    df = pd.DataFrame({"brent": brent, "gold": gold, "silver": silver})
    df = df.dropna()
    return df


# ---- Static tables (SWIFT payment shares & offshore RMB distribution) ----
# Source context: SWIFT RMB Tracker style summary, Dec-2025 snapshot used in the report.

def get_currency_shares():
    data = {
        "USD": 50.49, "EUR": 21.90, "GBP": 6.73, "CAD": 3.44, "JPY": 3.42,
        "CNY": 2.73, "HKD": 1.68, "AUD": 1.60, "SGD": 1.33, "CHF": 0.95,
    }
    return pd.Series(data, name="share_pct")


def get_offshore_rmb_distribution():
    data = {
        "Hong Kong": 75.9, "Others": 7.7, "United Kingdom": 7.2,
        "Singapore": 4.5, "France": 2.7, "United States": 2.0,
    }
    return pd.Series(data, name="share_pct")


EVENTS_BRENT = {
    "1973-10-01": "1973\nOPEC\nEmbargo",
    "1980-01-01": "1980\nIran-Iraq\nWar",
    "1986-01-01": "1986\nPrice\nCollapse",
    "1990-08-01": "1990\nGulf\nWar",
    "2008-07-01": "2008\nPeak",
    "2014-06-01": "2014\nShale\nGlut",
    "2020-03-01": "2020\nCOVID",
    "2022-02-01": "2022\nUkraine\nWar",
}

EVENTS_GOLD = {
    "1971-08-01": "1971\nNixon Shock",
    "1980-01-01": "1980\nPeak",
    "2008-10-01": "2008\nGFC",
    "2011-09-01": "2011\nPeak",
    "2020-08-01": "2020\nCOVID",
    "2022-03-01": "2022\nUkraine",
    "2024-10-01": "2024\nAll-Time\nHigh",
}

EVENTS_SILVER = {
    "1980-01-01": "1980\nHunt\nBrothers",
    "2008-10-01": "2008\nGFC",
    "2011-04-01": "2011\nPeak",
    "2020-03-01": "2020\nCOVID",
    "2021-02-01": "2021\nMeme\nSqueeze",
}

# Brent Crude Oil Analysis

## Chart 1 – Brent Crude Oil Prices with Major Events

In [ ]:
"""
Chart 1 - Brent Crude Oil Prices (1960-2026) with Major Events
Recreates: "Brent Crude Oil Prices (1960-2026) with Major Events"
"""
import matplotlib.pyplot as plt
import pandas as pd

df = get_market_frame()

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig, ax = plt.subplots(figsize=(11, 5.5))

ax.plot(df.index, df["brent"], color="#e8434f", linewidth=2, label="Brent Crude ($/bbl)")
ax.fill_between(df.index, df["brent"], color="#e8434f", alpha=0.12)

for date_str, label in EVENTS_BRENT.items():
    date = pd.Timestamp(date_str)
    if df.index.min() <= date <= df.index.max():
        ax.axvline(date, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.annotate(label, xy=(date, df["brent"].max() * 0.85),
                    ha="center", fontsize=8, color="#555555")

ax.set_title("Brent Crude Oil Prices (1960-2026) with Major Events", fontsize=15, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Price (USD per barrel)")
ax.legend(loc="upper left")
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("chart1_brent_trend.png", dpi=150)
plt.show()

## Chart 5 – Brent Crude Oil Annual Average Prices (Bar Chart)

In [ ]:
"""
Chart 5 - Brent Crude Oil Annual Average Prices (Bar Chart, 1960-2026)
"""
import matplotlib.pyplot as plt

df = get_market_frame()
annual = df["brent"].resample("YS").mean()

fig, ax = plt.subplots(figsize=(12, 5.5))
colors = plt.cm.autumn_r((annual - annual.min()) / (annual.max() - annual.min()))
ax.bar(annual.index.year, annual.values, color=colors, width=0.8)

ax.set_title("Brent Crude Oil - Annual Average Prices (1960-2026)", fontsize=15, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Average Price (USD per barrel)")
ax.set_xticks(range(1960, 2027, 5))

plt.tight_layout()
plt.savefig("chart5_brent_annual_bar.png", dpi=150)
plt.show()

# Gold Analysis

## Chart 2 – Gold Prices: Long-Term Trend & Safe-Haven Behaviour

In [ ]:
"""
Chart 2 - Gold Prices (1960-2026): Long-Term Trend & Safe-Haven Behaviour
"""
import matplotlib.pyplot as plt
import pandas as pd

df = get_market_frame()

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(df.index, df["gold"], color="#f4a259", linewidth=2)
ax.fill_between(df.index, df["gold"], color="#f4a259", alpha=0.12)

for date_str, label in EVENTS_GOLD.items():
    date = pd.Timestamp(date_str)
    if df.index.min() <= date <= df.index.max():
        ax.axvline(date, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.annotate(label, xy=(date, df["gold"].max() * 0.85), ha="center",
                    fontsize=8, color="#555555")

ax.set_title("Gold Prices (1960-2026) - Long-Term Trend & Safe-Haven Behaviour",
              fontsize=15, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Price (USD per troy oz)")
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("chart2_gold_trend.png", dpi=150)
plt.show()

## Chart B5 – Gold Price with Moving Averages

In [ ]:
"""
Chart B5 - Gold Price with 12-Month and 60-Month Moving Averages (1960-2026)
"""
import matplotlib.pyplot as plt

df = get_market_frame()
gold = df["gold"]
ma12 = gold.rolling(12).mean()
ma60 = gold.rolling(60).mean()

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(gold.index, gold.values, color="#c9c9c9", linewidth=1, label="Gold (monthly)")
ax.plot(ma12.index, ma12.values, color="#f4a259", linewidth=2, label="12-Month Moving Avg")
ax.plot(ma60.index, ma60.values, color="#8b3a3a", linewidth=2, label="60-Month Moving Avg")

ax.set_title("Gold Price with Moving Averages (1960-2026)", fontsize=15, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Price (USD per troy oz)")
ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig("chartB5_gold_moving_avg.png", dpi=150)
plt.show()

# Silver Analysis

## Chart 3 – Silver Prices: Peaks, Volatility & Industrial Demand

In [ ]:
"""
Chart 3 - Silver Prices (1960-2026): Peaks, Volatility & Industrial Demand
"""
import matplotlib.pyplot as plt
import pandas as pd

df = get_market_frame()

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(df.index, df["silver"], color="#4c78a8", linewidth=2)
ax.fill_between(df.index, df["silver"], color="#4c78a8", alpha=0.12)

for date_str, label in EVENTS_SILVER.items():
    date = pd.Timestamp(date_str)
    if df.index.min() <= date <= df.index.max():
        ax.axvline(date, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.annotate(label, xy=(date, df["silver"].max() * 0.55), ha="center",
                    fontsize=8, color="#555555")

ax.set_title("Silver Prices (1960-2026) - Peaks, Volatility & Industrial Demand",
              fontsize=15, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Price (USD per troy oz)")
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("chart3_silver_trend.png", dpi=150)
plt.show()

# Commodity Comparison

## Chart 4 – Gold vs Silver Price Comparison (Dual Y-Axis)

In [ ]:
"""
Chart 4 - Gold vs Silver Price Comparison (1960-2026), dual y-axis
"""
import matplotlib.pyplot as plt

df = get_market_frame()

fig, ax1 = plt.subplots(figsize=(11, 5.5))
ax2 = ax1.twinx()

l1, = ax1.plot(df.index, df["gold"], color="#f4a259", linewidth=2, label="Gold (left)")
l2, = ax2.plot(df.index, df["silver"], color="#4c78a8", linewidth=1.5, label="Silver (right)")

ax1.set_ylabel("Gold Price (USD/oz)", color="#f4a259")
ax2.set_ylabel("Silver Price (USD/oz)", color="#4c78a8")
ax1.tick_params(axis="y", labelcolor="#f4a259")
ax2.tick_params(axis="y", labelcolor="#4c78a8")
ax1.set_xlabel("Year")

ax1.set_title("Gold vs Silver Price Comparison (1960-2026)", fontsize=15, fontweight="bold")
ax1.legend(handles=[l1, l2], loc="upper left")

plt.tight_layout()
plt.savefig("chart4_gold_silver_combined.png", dpi=150)
plt.show()

## Chart B4 – Gold vs Silver Annual Average Prices (Scatter + Trendline)

In [ ]:
"""
Chart B4 - Gold vs Silver Annual Average Prices - Scatter Plot (1960-2026)
Includes a linear trendline and R^2 value.
"""
import matplotlib.pyplot as plt
import numpy as np

df = get_market_frame()
annual = df.resample("YS").mean()

x = annual["gold"].values
y = annual["silver"].values
years = annual.index.year

# Linear trendline
coeffs = np.polyfit(x, y, 1)
trend_x = np.array([x.min(), x.max()])
trend_y = np.polyval(coeffs, trend_x)
y_pred = np.polyval(coeffs, x)
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r_squared = 1 - ss_res / ss_tot

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(x, y, c=years, cmap="plasma", edgecolors="white", s=60, alpha=0.9)
ax.plot(trend_x, trend_y, color="red", linewidth=2, label=f"Trendline  R\u00b2={r_squared:.3f}")

ax.set_title("Gold vs Silver Annual Average Prices - Scatter Plot (1960-2026)",
              fontsize=14, fontweight="bold")
ax.set_xlabel("Gold Price (USD/oz)")
ax.set_ylabel("Silver Price (USD/oz)")
ax.legend(loc="upper left")

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Year")

plt.tight_layout()
plt.savefig("chartB4_gold_silver_scatter.png", dpi=150)
plt.show()

## Chart 9 – Monthly Price Volatility: Brent vs Gold vs Silver

In [ ]:
"""
Chart 9 - Monthly Price Volatility: Brent Oil vs Gold vs Silver (1960-2026)
Three stacked panels, month-over-month % change, gains vs losses color coded.
"""
import matplotlib.pyplot as plt

df = get_market_frame()
pct = df.pct_change().dropna() * 100

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
panels = [
    ("brent", "Brent Oil Monthly % Change", "#e8434f"),
    ("gold", "Gold Monthly % Change", "#f4a259"),
    ("silver", "Silver Monthly % Change", "#4c78a8"),
]

for ax, (col, title, color) in zip(axes, panels):
    values = pct[col]
    colors = [color if v >= 0 else "#9aa5ad" for v in values]
    ax.bar(values.index, values.values, color=colors, width=20)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("MoM % Change")

axes[-1].set_xlabel("Year")
fig.suptitle("Monthly Price Volatility: Brent Oil vs Gold vs Silver (1960-2026)",
             fontsize=15, fontweight="bold", y=1.0)

plt.tight_layout()
plt.savefig("chart9_monthly_volatility.png", dpi=150)
plt.show()

# Currency Analysis

## Chart 6 – Top 10 Currencies: Global Payment Share (SWIFT)

In [ ]:
"""
Chart 6 - Top 10 Currencies: Global Payment Share (Dec 2025, SWIFT)
"""
import matplotlib.pyplot as plt

shares = get_currency_shares().sort_values(ascending=True)

colors = []
for name in shares.index:
    if name == "USD":
        colors.append("#e8434f")
    elif name == "CNY":
        colors.append("#f4a259")
    else:
        colors.append("#9aa5ad")

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(shares.index, shares.values, color=colors)

for bar, value in zip(bars, shares.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{value:.2f}%", va="center", fontsize=10)

ax.set_title("Top 10 Currencies - Global Payment Share (Dec 2025, SWIFT)",
              fontsize=14, fontweight="bold")
ax.set_xlabel("Share (%)")
ax.set_xlim(0, shares.max() * 1.15)

from matplotlib.patches import Patch
legend_elems = [Patch(facecolor="#e8434f", label="USD"),
                Patch(facecolor="#f4a259", label="CNY/RMB"),
                Patch(facecolor="#9aa5ad", label="Other")]
ax.legend(handles=legend_elems, loc="lower right")

plt.tight_layout()
plt.savefig("chart6_currency_ranking.png", dpi=150)
plt.show()

## Chart 10 – USD/CNY Exchange Rate Trend

In [ ]:
"""
Chart 10 - USD/CNY Exchange Rate Trend (1994-2026)
"""
import matplotlib.pyplot as plt
import pandas as pd

series = get_usd_cny_series()

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(series.index, series.values, color="#2e7d5b", linewidth=2)
ax.fill_between(series.index, series.values, color="#2e7d5b", alpha=0.12)

d1 = pd.Timestamp("2005-07-01")
ax.axvline(d1, color="gray", linestyle="--", linewidth=0.8)
ax.annotate("2005\nDe-peg from\nUSD", xy=(d1, series.max() * 0.95),
            ha="center", fontsize=8, color="#555555")

d2 = pd.Timestamp("2015-08-01")
ax.axvline(d2, color="gray", linestyle="--", linewidth=0.8)
ax.annotate("2015\nRMB\nDevaluation", xy=(d2, series.max() * 0.85),
            ha="center", fontsize=8, color="#555555")

ax.set_title("USD/CNY Exchange Rate Trend (1994-2026)", fontsize=15, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("USD/CNY (RMB per US Dollar)")

plt.tight_layout()
plt.savefig("chart10_usd_cny_trend.png", dpi=150)
plt.show()

## Chart 7 – Offshore RMB Distribution by Economy

In [ ]:
"""
Chart 7 - Offshore RMB Distribution by Economy (Dec 2025)
"""
import matplotlib.pyplot as plt

shares = get_offshore_rmb_distribution()

colors = ["#e8434f", "#c8c8c8", "#f4a259", "#4c78a8", "#2e7d5b", "#9fe0e0"]

fig, ax = plt.subplots(figsize=(9, 9))
ax.pie(
    shares.values,
    labels=shares.index,
    autopct="%.1f%%",
    colors=colors,
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": "white", "linewidth": 1},
)
ax.set_title("Offshore RMB Distribution by Economy (Dec 2025)", fontsize=15, fontweight="bold")
ax.axis("equal")

plt.tight_layout()
plt.savefig("chart7_offshore_rmb.png", dpi=150)
plt.show()

# Correlation Analysis

## Chart B3 – Correlation Heatmap (Brent, Gold, Silver, USD/CNY)

In [ ]:
"""
Chart B3 - Correlation Heatmap: Brent Oil, Gold, Silver & USD/CNY (Monthly Returns)
"""
import matplotlib.pyplot as plt
import numpy as np

df = get_market_frame()
usdcny = get_usd_cny_series().reindex(df.index).interpolate()
df["usd_cny"] = usdcny

returns = df.pct_change().dropna()
corr = returns.corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)

labels = ["Brent Oil", "Gold", "Silver", "USD/CNY"]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                 color="black", fontsize=10)

ax.set_title("Correlation Heatmap: Brent Oil, Gold, Silver & USD/CNY\n(Monthly Returns, 1994-2026)",
              fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, label="Correlation Coefficient")

plt.tight_layout()
plt.savefig("chartB3_correlation_heatmap.png", dpi=150)
plt.show()

# Forecast

## Chart B6 – Brent Crude Oil Linear Trendline Forecast

In [ ]:
"""
Chart B6 - Brent Crude Oil - Linear Trendline Forecast (Estimate Only, Not a Prediction)
"""
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = get_market_frame()
hist = df["brent"].loc["2000-01-01":]

# Fit a simple linear trend on the last 24 months to project forward 24 months
recent = hist.iloc[-24:]
x_recent = np.arange(len(recent))
coeffs = np.polyfit(x_recent, recent.values, 1)

future_periods = 24
future_dates = pd.date_range(hist.index[-1], periods=future_periods + 1, freq="MS")[1:]
x_future = np.arange(len(recent), len(recent) + future_periods)
forecast = np.polyval(coeffs, x_future)
uncertainty = 15  # +/- $15 band, matches report annotation

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(hist.index, hist.values, color="#e8434f", linewidth=1.8, label="Historical Brent Oil")
ax.plot(future_dates, forecast, color="navy", linewidth=2.2, linestyle="--",
        label="Linear Forecast (estimate only)")
ax.fill_between(future_dates, forecast - uncertainty, forecast + uncertainty,
                 color="navy", alpha=0.15, label=f"Uncertainty Band (\u00b1${uncertainty})")
ax.axvline(hist.index[-1], color="gray", linestyle=":", linewidth=1)

ax.set_title("Brent Crude Oil - Linear Trendline Forecast (Estimate Only, Not a Prediction)",
              fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Price (USD/bbl)")
ax.legend(loc="upper left")

ax.text(0.02, 0.05,
        "\u26a0 Forecast is a simple linear estimate only.\n"
        "Actual prices depend on geopolitical, supply, and demand factors.",
        transform=ax.transAxes, fontsize=9, style="italic",
        bbox=dict(boxstyle="round", facecolor="#fff9c4", edgecolor="#cccccc"))

plt.tight_layout()
plt.savefig("chartB6_brent_forecast.png", dpi=150)
plt.show()

# Dashboard

## Chart 11 – Global Commodity & Currency Market Dashboard

In [ ]:
"""
Chart 11 - Global Commodity & Currency Market Dashboard (1960-2026)
2x2 dark-themed dashboard: Brent, Gold, Silver trends + currency share bar chart.
"""
import matplotlib.pyplot as plt

df = get_market_frame()
shares = get_currency_shares().sort_values(ascending=True).tail(6)

plt.style.use("dark_background")
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.patch.set_facecolor("#0d1b2a")

# Brent
ax = axes[0, 0]
ax.plot(df.index, df["brent"], color="#e8434f", linewidth=1.8)
ax.fill_between(df.index, df["brent"], color="#e8434f", alpha=0.15)
ax.set_title("Brent Crude Oil", fontweight="bold")
ax.set_ylabel("USD/bbl")

# Gold
ax = axes[0, 1]
ax.plot(df.index, df["gold"], color="#f4a259", linewidth=1.8)
ax.fill_between(df.index, df["gold"], color="#f4a259", alpha=0.15)
ax.set_title("Gold", fontweight="bold")
ax.set_ylabel("USD/oz")

# Silver
ax = axes[1, 0]
ax.plot(df.index, df["silver"], color="#4c78a8", linewidth=1.8)
ax.fill_between(df.index, df["silver"], color="#4c78a8", alpha=0.15)
ax.set_title("Silver", fontweight="bold")
ax.set_ylabel("USD/oz")

# Currency shares
ax = axes[1, 1]
colors = ["#e8434f" if n == "USD" else ("#f4a259" if n == "CNY" else "#9aa5ad")
          for n in shares.index]
ax.barh(shares.index, shares.values, color=colors)
ax.set_title("Top Currency Payment Shares (Dec 2025)", fontweight="bold")
ax.set_xlabel("Share (%)")

fig.suptitle("Global Commodity & Currency Market Dashboard  |  1960-2026",
             fontsize=17, fontweight="bold", color="white")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("chart11_dashboard.png", dpi=150, facecolor=fig.get_facecolor())
plt.show()